# CellPhoneDB v5 analysis per patient level

**Goal.** Find ligand–receptor interactions between cell states that recur across on-treatment
patients, rather than interactions driven by a single patient.

**Input.** Log-normalised `.h5ad` per patient (`cellphoneDB_adata_per_patient/`),
cell states in `cell_state_x`.

**Output.** `cpdb_per_patient/<patient>/statistical_analysis_*.txt`;
`cpdb_adata_per_patient/consensus_*.h5ad`; plots in `cpdb_adata_per_patient/plots/`.

In [ ]:
# !pip install --quiet scanpy cellphonedb ktplotspy seaborn
# %cd <project>/cpdb/cellphoneDB_v5_plot/OT

In [ ]:
import glob
import os
from importlib.metadata import version

import anndata as ad
import ktplotspy as kpy
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
from cellphonedb.src.core.methods import cpdb_statistical_analysis_method
from plotnine import facet_wrap

print({p: version(p) for p in ["cellphonedb", "scanpy", "anndata", "ktplotspy"]})

## Config

In [ ]:
group = "OT" # on treatment (partial response + stable disease) # note: change this line to TN or TKIR patients 

patients = ["P001", "P002", "P003", "P005", "P006", "P008", "P009", "P011", "P012", "P013"] # note: change this line to TN or TKIR patients 

lognorm_dir = "../../cellphoneDB_adata_per_patient"   # <patient>_log_normalized.h5ad (notebook)
cpdb_db_path = "./cellphonedb.zip"                    # CellPhoneDB v5 database
inputs_dir = "./cpdb_inputs_per_patient"              # counts + meta given to CellPhoneDB
results_dir = "./cpdb_per_patient"                    # CellPhoneDB output per patient
consensus_dir = "./cpdb_adata_per_patient"            # per-patient + consensus AnnData
plot_dir = f"{consensus_dir}/plots"
label_col = "cell_state_x"


for d in [inputs_dir, results_dir, consensus_dir, plot_dir]:
    os.makedirs(d, exist_ok=True)

## 1. Run CellPhoneDB v5 per patient

Input check: if the file carries `layers['raw_counts']`, `X` must equal log1p(counts per 10k) of
those counts (guards against double log-normalisation).

In [ ]:
# --- per-patient input filters (applied before CellPhoneDB) ---
min_celltype_fraction = 0.005           # keep states with > 0.5% of the patient's cells
min_genes_per_cell = 200
min_cells_per_gene = 3


# --- CellPhoneDB v5 statistical analysis ---
cpdb_params = dict(
    counts_data="hgnc_symbol",
    active_tfs_file_path=None,
    microenvs_file_path=None,
    score_interactions=True,
    iterations=1000,
    threshold=0.1,                      # min fraction of cells in a state expressing the gene
    threads=16,
    debug_seed=42,
    result_precision=3,
    pvalue=0.05,
    subsampling=False,
    subsampling_log=False,
    subsampling_num_pc=100,
    subsampling_num_cells=1000,
    separator="|",
    debug=False,
)

In [ ]:
def prepare_inputs(patient):
    a = sc.read_h5ad(f"{lognorm_dir}/{patient}_log_normalized.h5ad")
    if "raw_counts" in a.layers:
        ref = sc.AnnData(a.layers["raw_counts"][:200].copy())
        sc.pp.normalize_total(ref, target_sum=1e4)
        sc.pp.log1p(ref)
        assert np.allclose(ref.X.toarray(), a.X[:200].toarray(), atol=1e-3), f"{patient}: X is not log1p(CP10k)"

    counts = a.obs[label_col].value_counts()
    min_cells = int(a.n_obs * min_celltype_fraction)
    kept_states = counts[counts > min_cells].index
    a = a[a.obs[label_col].isin(kept_states)].copy()
    sc.pp.filter_cells(a, min_genes=min_genes_per_cell)
    sc.pp.filter_genes(a, min_cells=min_cells_per_gene)

    out = f"{inputs_dir}/{patient}"
    os.makedirs(out, exist_ok=True)
    sc.AnnData(a.X, obs=pd.DataFrame(index=a.obs_names), var=pd.DataFrame(index=a.var_names)).write_h5ad(
        f"{out}/counts.h5ad", compression="gzip")
    pd.DataFrame({"Cell": a.obs_names, "cell_type": a.obs[label_col].astype(str).values}).to_csv(
        f"{out}/meta.tsv", sep="\t", index=False)
    return {"patient": patient, "cells": a.n_obs, "genes": a.n_vars, "states_kept": len(kept_states),
            "states_dropped": ", ".join(sorted(set(counts[counts > 0].index) - set(kept_states))),
            "min_cells_per_state": min_cells}


run_log = []
for patient in patients:
    run_log.append(prepare_inputs(patient))
    cpdb_statistical_analysis_method.call(
        cpdb_file_path=cpdb_db_path,
        meta_file_path=f"{inputs_dir}/{patient}/meta.tsv",
        counts_file_path=f"{inputs_dir}/{patient}/counts.h5ad",
        output_path=f"{results_dir}/{patient}",
        output_suffix=patient,
        **cpdb_params,
    )

run_log = pd.DataFrame(run_log).set_index("patient")
run_log.to_csv(f"{results_dir}/run_log.csv")
run_log

## 2. Per-patient heatmap (number of significant interactions per cell pair)

In [ ]:
# --- consensus ---
alpha = 0.05
interaction_prevalence = 1.0            # interaction significant in this fraction of patients where tested
slot_thresholds = {"strict": 1.0, "0.5_strict": 0.5}   # per cell pair: fraction of patients significant
min_patients_per_slot = 1               # cell pair must be tested in >= this many patients (1 = no filter)

def read_result(patient, kind):
    files = sorted(glob.glob(f"{results_dir}/{patient}/statistical_analysis_{kind}_*.txt"))
    assert files, f"{patient}: no {kind} file"
    return pd.read_csv(files[-1], sep="\t")

for patient in patients:
    p = kpy.plot_cpdb_heatmap(pvals=read_result(patient, "pvalues"), alpha=alpha, figsize=(9, 9),
                              title=f"{patient}: sum of significant interactions")
    p.fig.savefig(f"{plot_dir}/{patient}_cpdb_heatmap.pdf", bbox_inches="tight")
    plt.show()

## 3. Per-patient result AnnData

One AnnData per patient: observations = cell pairs (`sender|receiver`), variables = interactions,
layers `means` and `pvalues`.

In [ ]:
var_cols = ["id_cp_interaction", "interacting_pair", "partner_a", "partner_b", "gene_a", "gene_b",
            "secreted", "receptor_a", "receptor_b", "annotation_strategy", "is_integrin",
            "directionality", "classification"]
sep = cpdb_params["separator"]


def to_adata(means, pvals):
    pairs = [c for c in means.columns if sep in c]
    var = means[[c for c in var_cols if c in means.columns]].set_index("id_cp_interaction", drop=False)
    obs = pd.DataFrame(index=pairs)
    obs["sender"], obs["receiver"] = zip(*(p.split(sep) for p in pairs))
    a = ad.AnnData(X=means[pairs].T.values, obs=obs, var=var)
    a.layers["means"] = means[pairs].T.values
    a.layers["pvalues"] = pvals.set_index("id_cp_interaction").loc[var.index, pairs].T.values
    return a


adata_dict = {}
for patient in patients:
    adata_dict[patient] = to_adata(read_result(patient, "means"), read_result(patient, "pvalues"))
    adata_dict[patient].write_h5ad(f"{consensus_dir}/{patient}_adata_cpdb.h5ad")
pd.DataFrame({p: {"cell_pairs": a.n_obs, "interactions": a.n_vars} for p, a in adata_dict.items()}).T

## 4. Cross-patient consensus

- **Interaction filter:** an interaction is kept if it is significant (in ≥ 1 cell pair) in at least
  `interaction_prevalence` of the patients in which CellPhoneDB tested it.
- **Per cell pair × interaction:** `means` = mean across patients where the pair exists;
  `fraction_significant` = significant patients / patients where the pair exists;
  `n_patients` = patients where the pair exists; `pvalues` = minimum p-value across patients.

In [ ]:
def layer_df(a, layer):
    return pd.DataFrame(a.layers[layer], index=a.obs_names, columns=a.var_names)


# interaction-level prevalence
sig_any = pd.DataFrame({p: (layer_df(a, "pvalues") < alpha).any(axis=0) for p, a in adata_dict.items()})
n_tested = sig_any.notna().sum(axis=1)
frac_patients = sig_any.fillna(False).sum(axis=1) / n_tested
selected = frac_patients.index[frac_patients >= interaction_prevalence]
print(f"{len(sig_any)} interactions tested; {(n_tested == len(patients)).sum()} tested in all "
      f"{len(patients)} patients; {len(selected)} kept at prevalence >= {interaction_prevalence}")

# cell pair x interaction summaries across patients
means_all = pd.concat([layer_df(a, "means") for a in adata_dict.values()]).groupby(level=0).mean()[selected]
sig_sum, n_present, p_min = 0, 0, None
for a in adata_dict.values():
    pv = layer_df(a, "pvalues").reindex(index=means_all.index, columns=selected)
    sig_sum = sig_sum + (pv < alpha).astype(float)
    n_present = n_present + pv.notna().astype(float)
    p_min = pv if p_min is None else np.fmin(p_min, pv)
fraction = (sig_sum / n_present.replace(0, np.nan)).fillna(0)

var = (pd.concat([a.var for a in adata_dict.values()]).drop_duplicates("id_cp_interaction")
       .set_index("id_cp_interaction", drop=False).loc[selected].astype(str))
obs = pd.DataFrame(index=means_all.index)
obs["sender"], obs["receiver"] = zip(*(p.split(sep) for p in obs.index))

adata_consensus = ad.AnnData(X=means_all.values.astype(np.float32), obs=obs, var=var)
adata_consensus.layers["means"] = means_all.values.astype(np.float32)
adata_consensus.layers["fraction_significant"] = fraction.values.astype(np.float32)
adata_consensus.layers["n_patients"] = n_present.values.astype(np.float32)
adata_consensus.layers["pvalues"] = p_min.fillna(1.0).values.astype(np.float32)
adata_consensus.write_h5ad(f"{consensus_dir}/consensus_ALL_cellpairs_fraction_significant.h5ad")
adata_consensus

## 5. Compartments and per-slot recurrence filters

In [ ]:
# --- compartments and plots ---
tumour_states = ["1_Epithelial baseline", "2_Differentiated epithelial", "3_Devp_stem-like",
                 "4_Inflammatory EMT", "5_Neurod_EMT-like", "6_Proliferative"]
immune_states = ["Macrophage INHBAhigh", "Macrophage STAB1high", "Macrophage CD36high",
                 "cDC2", "cDC1_activatedDC",
                 "CD4 T naive", "CD4 Tem", "CD4 T regulatory",
                 "CD8 T naive", "CD8 Tem", "CD8 Tem_ENTPD1high",
                 "NK cell", "Cytotoxic NK cell",
                 "B cell", "B cell dividing", "Plasma cell"]
heatmap_order = tumour_states + immune_states
heatmap_vmax = 25
top_n = 20

s, r = adata_consensus.obs["sender"], adata_consensus.obs["receiver"]
compartments = {
    "tumour_immune": (s.isin(tumour_states) & r.isin(immune_states)) | (s.isin(immune_states) & r.isin(tumour_states)),
    "tumour_tumour": s.isin(tumour_states) & r.isin(tumour_states),
    "immune_immune": s.isin(immune_states) & r.isin(immune_states),
}
parts = []
for name, mask in compartments.items():
    a = adata_consensus[mask.to_numpy()].copy()
    a.obs["compartment"] = name
    parts.append(a)
adata_combined = ad.concat(parts, merge="same")
adata_combined.write_h5ad(f"{consensus_dir}/consensus_compartments.h5ad")
print(adata_combined.obs["compartment"].value_counts().to_dict())


def recurrence_filter(a, threshold):
    keep = (a.layers["fraction_significant"] >= threshold) & (a.layers["n_patients"] >= min_patients_per_slot)
    out = a.copy()
    out.layers["means"] = np.where(keep, a.layers["means"], 0).astype(np.float32)
    out.layers["pvalues"] = np.where(keep, a.layers["pvalues"], 1).astype(np.float32)
    out.layers["fraction_significant"] = np.where(keep, a.layers["fraction_significant"], 0).astype(np.float32)
    out.X = out.layers["means"]
    out = out[keep.any(axis=1) & (out.X.max(axis=1) > 0), keep.any(axis=0) & (out.X.max(axis=0) > 0)].copy()
    return out


filtered = {"all": adata_combined}
for label, thr in slot_thresholds.items():
    filtered[label] = recurrence_filter(adata_combined, thr)
    filtered[label].write_h5ad(f"{consensus_dir}/consensus_compartments_{label}.h5ad")

pd.DataFrame({k: {"cell_pairs": a.n_obs, "interactions": a.n_vars,
                  **a.obs["compartment"].value_counts().to_dict()} for k, a in filtered.items()}).T

## 6. Summary heatmaps

Number of interactions per sender → receiver pair: significant in ≥ 1 patient (`all`), or
passing the per-slot recurrence threshold (`strict`, `0.5_strict`).

In [ ]:
titles = {"all": "significant in >= 1 patient", "strict": "significant in 100% of patients",
          "0.5_strict": "significant in >= 50% of patients"}
for label, a in filtered.items():
    count = pd.Series((a.layers["fraction_significant"] > 0).sum(axis=1), index=a.obs_names)
    mat = (pd.DataFrame({"sender": a.obs["sender"], "receiver": a.obs["receiver"], "n": count})
           .pivot_table(index="sender", columns="receiver", values="n", aggfunc="sum", fill_value=0)
           .reindex(index=heatmap_order, columns=heatmap_order, fill_value=0))
    fig, ax = plt.subplots(figsize=(12, 10))
    sns.heatmap(mat, cmap="Reds", vmax=heatmap_vmax, annot=True, fmt=".0f", linewidths=0.5, linecolor="grey",
                cbar_kws={"label": f"No. interactions ({titles[label]})"}, ax=ax)
    ax.set(title=f"{group}: recurrent L-R interactions, {titles[label]}", xlabel="Receiver", ylabel="Sender")
    plt.savefig(f"{plot_dir}/heatmap_cellpair_{label}.pdf", bbox_inches="tight") #Figure 6
    plt.show()

## 7. Top tumour-immune interactions

Interactions ranked by Σ over tumour–immune cell pairs of (mean expression × fraction of patients
significant). Dot plots with ktplotspy: all tumour states, faceted by classification, and one per
tumour state.

In [ ]:
def ktpy_inputs(a):
    m = layer_df(a, "means").T
    p = layer_df(a, "pvalues").T
    for df in (m, p):
        df.index = a.var["interacting_pair"].values
        for col in reversed([c for c in var_cols if c in a.var]):
            df.insert(0, col, a.var[col].values)
    return m, p


tumour_str, immune_str = "|".join(tumour_states), "|".join(immune_states)
for label, a in filtered.items():
    out = f"{plot_dir}/{label}"
    os.makedirs(out, exist_ok=True)
    ti = a[a.obs["compartment"] == "tumour_immune"].copy()
    score = pd.Series((ti.layers["means"] * ti.layers["fraction_significant"]).sum(axis=0), index=ti.var_names)
    top = score.sort_values(ascending=False).head(top_n)
    table = ti.var.loc[top.index, ["interacting_pair", "gene_a", "gene_b", "secreted"]].assign(weighted_score=top.values)
    table.to_csv(f"{out}/top{top_n}_interactions_tumour_immune.csv", index=False)
    genes = table["interacting_pair"].tolist()

    means_k, pvals_k = ktpy_inputs(ti)
    specs = [("ktpy_tumour_immune_top20", tumour_str, (16, 8), None)]
    specs.append(("ktpy_tumour_immune_by_classification", tumour_str, (14, 10), "classification"))
    specs += [(f"ktpy_{t.replace(' ', '_')}_vs_immune", t, (14, 6), None) for t in tumour_states]
    for name, ct1, size, facet in specs:
        try:
            p = kpy.plot_cpdb(adata=ti, cell_type1=ct1, cell_type2=immune_str, means=means_k, pvals=pvals_k,
                              celltype_key="sender", genes=genes, figsize=size, title=f"{name} ({label})",
                              max_size=6, highlight_size=0.75, standard_scale=True)
            if facet:
                p = p + facet_wrap(f"~ {facet}", ncol=1)
            p.save(f"{out}/{name}.pdf", width=size[0], height=size[1])
        except Exception as e:                              # e.g. no significant slot for this tumour state
            print(f"{label}/{name}: not plotted ({e})")
    print(label, "->", out)
table